Imports y carga de datos

In [1]:
import polars as pl
import pandas as pd
import time
import matplotlib.pyplot as plt

file_path = '../taxi_filtrado.parquet'
df_full_pl = pl.read_parquet(file_path)
df_full_pd = pd.read_parquet(file_path)

#   Definición de pipelines

In [2]:
def run_polars_pipeline(df):
    t0 = time.time()
    df = df.filter((pl.col('trip_distance') > 0) & (pl.col('total_amount') > 0))
    agg = df.group_by('passenger_count').agg(pl.col('total_amount').mean().alias('avg_total_amount'))
    df = df.join(agg, on='passenger_count', how='left')
    df = df.with_columns((pl.col('total_amount') / pl.col('trip_distance')).alias('cost_per_mile'))
    return time.time() - t0

def run_pandas_pipeline(df):
    t0 = time.time()
    df = df[(df['trip_distance'] > 0) & (df['total_amount'] > 0)]
    agg = df.groupby('passenger_count')['total_amount'].mean().reset_index(name='avg_total_amount')
    df = df.merge(agg, on='passenger_count', how='left')
    df['cost_per_mile'] = df['total_amount'] / df['trip_distance']
    return time.time() - t0

# Experimento de escalabilidad (25%, 50%, 75%, 100%)

In [ ]:
percentages = [0.25, 0.50, 0.75, 1.00]
results = []

for p in percentages:
    df_sub_pl = df_full_pl.sample(fraction=p, seed=42)
    df_sub_pd = df_full_pd.sample(frac=p, random_state=42)

    t_pl = run_polars_pipeline(df_sub_pl)
    t_pd = run_pandas_pipeline(df_sub_pd)

    results.append({
        'Porcentaje': f'{int(p*100)}%',
        'Filas': len(df_sub_pl),
        'Pandas_s': t_pd,
        'Polars_s': t_pl,
        'Speedup': t_pd / t_pl
    })

df_exp = pl.DataFrame(results)
print(df_exp)

# Gráficas de resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = df_exp['Porcentaje'].to_list()

# Tiempos
axes[0].plot(x, df_exp['Pandas_s'].to_numpy(), marker='o', label='Pandas', color='red')
axes[0].plot(x, df_exp['Polars_s'].to_numpy(), marker='s', label='Polars', color='blue')
axes[0].set_title('Tiempo Total vs Tamaño del Dataset')
axes[0].set_xlabel('Porcentaje del Dataset')
axes[0].set_ylabel('Tiempo (segundos)')
axes[0].legend()
axes[0].grid(True)

# Speedup
axes[1].bar(x, df_exp['Speedup'].to_numpy(), color='green', alpha=0.7)
axes[1].set_title('Speedup (Pandas / Polars)')
axes[1].set_xlabel('Porcentaje del Dataset')
axes[1].set_ylabel('Speedup (x veces más rápido)')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined